# Books Catalog Extractor

This notebook demonstrates a web scraping workflow using Python.

**What this demo covers:**
- Respectful scraping practices
- HTTP requests with headers, timeout, retries, and backoff
- HTML parsing with BeautifulSoup
- Pagination handling
- Structured data extraction
- Data validation and cleaning
- Export to CSV and JSON
- Basic reporting using pandas



## 1. Install and Import Required Libraries

The stack below is common for lightweight scraping jobs:
- `requests` for HTTP calls
- `BeautifulSoup` for HTML parsing
- `pandas` for data processing/export
- `urllib3 Retry` for resilient requests

In [2]:
!pip -q install beautifulsoup4 lxml pandas requests

In [1]:
import time
import re
import json
from dataclasses import dataclass, asdict
from typing import List, Optional
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## 2. Configuration

This keeps scraper settings in one place, similar to how production code should be configured.

In [6]:
BASE_URL = "https://books.toscrape.com/"
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; PortfolioScraper/1.0; +https://github.com/summanbahadur-ai)",
    "Accept-Language": "en-US,en;q=0.9",
}

REQUEST_TIMEOUT = 15
REQUEST_DELAY_SECONDS = 0.5
MAX_PAGES = 5  # Keep demo small.

## 3. Create a Reliable HTTP Session

A good scraper should not fail immediately because of a temporary network issue.  
This session retries failed requests with a small backoff.

In [7]:
def create_session() -> requests.Session:
    session = requests.Session()

    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(HEADERS)
    return session

session = create_session()

## 4. Define the Output Data Model

Using a dataclass makes the extracted data consistent and easier to validate.

In [8]:
@dataclass
class BookItem:
    title: str
    price_gbp: float
    availability: str
    rating: str
    product_url: str
    image_url: str
    category: Optional[str] = None


def clean_price(price_text: str) -> float:
    """Convert price text like '£51.77' to 51.77."""
    cleaned = re.sub(r"[^0-9.]", "", price_text)
    return float(cleaned) if cleaned else 0.0


def clean_text(value: str) -> str:
    """Normalize whitespace."""
    return re.sub(r"\s+", " ", value).strip()

## 5. Fetch and Parse a Page

This function handles the HTTP request and returns a BeautifulSoup object.

In [9]:
def fetch_page(url: str) -> BeautifulSoup:
    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")

## 6. Extract Book Data from One Page

The parser is isolated from the network request.  
This makes the scraper easier to test and maintain.

In [10]:
def parse_books_from_page(soup: BeautifulSoup, page_url: str) -> List[BookItem]:
    books: List[BookItem] = []
    product_cards = soup.select("article.product_pod")

    for card in product_cards:
        title = card.select_one("h3 a")["title"]
        relative_product_url = card.select_one("h3 a")["href"]
        product_url = urljoin(page_url, relative_product_url)

        relative_image_url = card.select_one("img")["src"]
        image_url = urljoin(page_url, relative_image_url)

        price_text = card.select_one("p.price_color").get_text(strip=True)
        price_gbp = clean_price(price_text)

        availability = clean_text(card.select_one("p.instock.availability").get_text())

        rating_classes = card.select_one("p.star-rating").get("class", [])
        rating = next((cls for cls in rating_classes if cls != "star-rating"), "Not Rated")

        books.append(
            BookItem(
                title=title,
                price_gbp=price_gbp,
                availability=availability,
                rating=rating,
                product_url=product_url,
                image_url=image_url,
            )
        )

    return books

## 7. Handle Pagination

This scraper follows the **next** button until the limit is reached or no more pages are available.

In [11]:
def get_next_page_url(soup: BeautifulSoup, current_url: str) -> Optional[str]:
    next_link = soup.select_one("li.next a")
    if not next_link:
        return None
    return urljoin(current_url, next_link["href"])


def scrape_books(start_url: str, max_pages: int = MAX_PAGES) -> List[BookItem]:
    all_books: List[BookItem] = []
    current_url = start_url
    page_number = 1

    while current_url and page_number <= max_pages:
        print(f"Scraping page {page_number}: {current_url}")
        soup = fetch_page(current_url)

        page_books = parse_books_from_page(soup, current_url)
        all_books.extend(page_books)

        current_url = get_next_page_url(soup, current_url)
        page_number += 1
        time.sleep(REQUEST_DELAY_SECONDS)

    return all_books

## 8. Run the Scraper

In [12]:
books = scrape_books(START_URL, MAX_PAGES)
print(f"Total books scraped: {len(books)}")
books[:3]

Scraping page 1: https://books.toscrape.com/catalogue/page-1.html
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Total books scraped: 100


[BookItem(title='A Light in the Attic', price_gbp=51.77, availability='In stock', rating='Three', product_url='https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', image_url='https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg', category=None),
 BookItem(title='Tipping the Velvet', price_gbp=53.74, availability='In stock', rating='One', product_url='https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html', image_url='https://books.toscrape.com/media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg', category=None),
 BookItem(title='Soumission', price_gbp=50.1, availability='In stock', rating='One', product_url='https://books.toscrape.com/catalogue/soumission_998/index.html', image_url='https://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg', category=None)]

## 9. Convert to DataFrame and Validate Data

A production scraper should check for missing values, duplicates, and suspicious records.

In [13]:
df = pd.DataFrame([asdict(book) for book in books])

print("Rows:", len(df))
print("Duplicate product URLs:", df["product_url"].duplicated().sum())
print("Missing values per column:")
print(df.isna().sum())

df.head()

Rows: 100
Duplicate product URLs: 0
Missing values per column:
title             0
price_gbp         0
availability      0
rating            0
product_url       0
image_url         0
category        100
dtype: int64


,title,price_gbp,availability,rating,product_url,image_url,category
0,A Light in the Attic,51.77,In stock,Three,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/media/cache/2c/da/2...,None
1,Tipping the Velvet,53.74,In stock,One,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/media/cache/26/0c/2...,None
2,Soumission,50.10,In stock,One,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...,None
3,Sharp Objects,47.82,In stock,Four,https://books.toscrape.com/catalogue/sharp-obj...,https://books.toscrape.com/media/cache/32/51/3...,None
4,Sapiens: A Brief History of Humankind,54.23,In stock,Five,https://books.toscrape.com/catalogue/sapiens-a...,https://books.toscrape.com/media/cache/be/a5/b...,None


In [14]:
# Basic quality checks
assert not df.empty, "No data scraped."
assert df["product_url"].is_unique, "Duplicate product URLs found."
assert (df["price_gbp"] >= 0).all(), "Negative price found."

print("Data validation passed.")

Data validation passed.


## 10. Simple Analysis

This section shows that the scraped data can immediately be used for reporting.

In [15]:
summary = {
    "total_books": len(df),
    "average_price_gbp": round(df["price_gbp"].mean(), 2),
    "min_price_gbp": round(df["price_gbp"].min(), 2),
    "max_price_gbp": round(df["price_gbp"].max(), 2),
}
summary

{'total_books': 100,
 'average_price_gbp': np.float64(34.56),
 'min_price_gbp': 10.16,
 'max_price_gbp': 58.11}

In [16]:
rating_summary = df.groupby("rating").agg(
    count=("title", "count"),
    average_price_gbp=("price_gbp", "mean")
).reset_index()

rating_summary["average_price_gbp"] = rating_summary["average_price_gbp"].round(2)
rating_summary

,rating,count,average_price_gbp
0,Five,19,30.01
1,Four,18,33.98
2,One,22,35.52
3,Three,22,36.84
4,Two,19,35.91


## 11. Export Results

The output can be shared with a client or loaded into another backend service.

In [17]:
CSV_FILE = "scraped_books.csv"
JSON_FILE = "scraped_books.json"

df.to_csv(CSV_FILE, index=False)
df.to_json(JSON_FILE, orient="records", indent=2)

print(f"Saved: {CSV_FILE}")
print(f"Saved: {JSON_FILE}")

Saved: scraped_books.csv
Saved: scraped_books.json


## Portfolio Note

This notebook is a demonstration of my approach to web scraping: reliable requests, clean parsing, data validation, and export-ready output.

For Java/Spring Boot projects, the same scraper can be integrated in two ways:

- **Option 1:** Fix or improve the existing Java-based scraper inside the Spring Boot backend.
- **Option 2:** Create a separate Python scraper service and connect it to the Java backend through REST/API, message queue, or scheduled jobs.
